# BP6 Gate 5 — Decision / GenAI Layer & Reporting
**Customer360 Navigator Enterprise Suite — GenAI Resolution Assistant**

## Provider pivot (2026-09-24, same day as the original decision)

This gate was first built against the Anthropic Claude API, per your initial 2026-09-24 decision.
Later the same day you asked for a free-of-cost path instead. Anthropic's own **Claude for
Startups** credit program was checked and ruled out — its stated eligibility requires institutional
equity funding and a company founded within the last four years; this project is a personal
portfolio build during a job search, not a funded company, so it does not meet that bar. You then
chose **Google's Gemini API** specifically because it has a genuine, standing no-cost free tier
(the Flash-family models), reachable with just a Google AI Studio API key — no funded billing
account required. This gate has been rebuilt end-to-end against the official `google-genai` SDK.
Every design principle below (grounding, citation enforcement, UDAAP review, human-in-the-loop) is
unchanged — only the model provider and its SDK-specific plumbing changed.

## This is BP6's first gate with a real external API call

Every earlier gate (1–4) was prep-only, and each structurally verified **zero** GenAI SDK modules
loaded. This gate is the opposite: it is the one place in the entire Customer360 Navigator suite
that makes a real, live call to an external GenAI provider — the Google Gemini API. Per your
pluggable-build decision, the real call is fully wired end-to-end, but if `GEMINI_API_KEY` is not
set in the environment, the notebook raises a clear `MissingApiKeyError` naming exactly what to do,
and writes **nothing** — no artifact, no config block — rather than fabricating or stubbing a
response. Sandbox-verified: with no key set, Sections 1–6 run cleanly against real data and
Section 7 stops with that exact disclosed error, leaving the config and artifacts directory
byte-for-byte unchanged (md5-confirmed before/after) — this was previously proven true for the
Anthropic-based version and has now been re-proven true for this Gemini-based rebuild.

## Why this gate looks different from BP1-5's own Gate 5

The Master Plan's generic Gate 5 row ("Decision / GenAI Layer & Reporting" — output: "Grounded
GenAI output / decision-engine score with reason codes"; exit criteria: "Every claim carries a
citation/evidence field; every score has reason codes"; compliance touchpoint: "UDAAP language
review on customer-facing text; NIST AI RMF Measure/Manage check on BP6") is written to cover both
BP6 and BP7 at once. Master Plan paragraph 44 makes the split explicit: **"reason codes" is BP7's
mechanism** (transparent decision rules with recorded reason codes and thresholds), not BP6's —
paragraph 43 gives BP6's own real shape: "retrieve structured evidence (from BP1–BP5 outputs),
summarize and recommend a next action with citations/evidence fields on every claim; human-in-the-
loop approval gate before any recommendation is treated as final." So this gate does not invent a
"reason codes" analog for BP6 — its real exit criterion is the citation requirement alone, already
fully specified.

Paragraph 117 states the real architecture directly: **"Use retrieval/grounding and deterministic
templates around GenAI (BP6) — no ungrounded generation is ever presented as a recommendation."**
This gate implements that literally, not as a design aspiration, regardless of which provider sits
behind the call:

1. **The model is never allowed to invent its own evidence.** It receives a real, pre-assembled
   evidence bundle (Section 4) — 28 real citation records this run, each a real top-level scalar
   field read directly from BP1–BP5's own real Gate 7 executive rollup manifests (schema-agnostic:
   BP1–3 use `champion_model`, BP4 uses `champion_pipeline`, BP5 uses neither — each BP's own real
   schema is read as-is, never assumed uniform, per BP6's own stated design principle).
2. **The model's output is never trusted on its own word.** After generation, every `[EV-...]`
   citation tag it actually used is structurally checked against the real bundle
   (`validate_citations_in_generated_text`) — an invented or mis-cited tag fails the check, and a
   failed check means the recommendation is never written to disk, not even flagged.
3. **UDAAP customer-facing language review** — BP6's own adaptation of the per-sentence,
   quote-masking scanner pattern BP5 Gate 5 established, but targeting deceptive/misleading
   customer-facing language (guarantees, absolute promises, urgency pressure) rather than BP5's
   causal-claim language, since BP6's real risk is different from BP5's.
4. **NIST AI RMF risk category** — populated with a real, structurally-computed value for the
   first time (Gate 1's own policy.json carried `TBD_PENDING_FIRST_REAL_GATE5_GENAI_OUTPUT` since
   Gate 1). No official NIST-published LOW/MEDIUM/HIGH enum exists for this — Master Plan Section 9
   only requires "a documented risk category" — so this gate computes one from real, disclosed
   run facts (citation check passed, UDAAP check passed, human-in-the-loop enforced, auto-apply
   forbidden, PII masking applied upstream). A live, real, customer-facing external LLM call is
   never categorized LOW in this project, regardless of how many mitigations pass.
5. **Human-in-the-loop, never auto-applied.** The final artifact's `approval_status` is always
   `"PENDING_HUMAN_REVIEW"` and `auto_applied` is always `False` — this notebook stages a
   recommendation for a person to review; it never sends, applies, or finalizes anything.

## What this gate does

1. **Real evidence bundle** (Section 4) — `retrieve_headline_evidence_bundle`
   (`src/genai/bp6_grounded_generation.py`) opens BP1–5's real Gate 7 manifests. Unchanged by the
   provider pivot.
2. **Real customer message selection** (Section 5) — one real, PII-clear row from Gate 2's own
   real, already-screened narrative CSV, restricted to Gate 3/4's own real cross-corpus-hit
   buckets (read from Gate 4's own explainability trace, never hardcoded) — deterministic, seeded.
   Unchanged by the provider pivot.
3. **Deterministic grounded prompt** (Section 6) — instructs the model to cite only the real
   evidence given, never guarantee an outcome, never claim an action already taken. Unchanged by
   the provider pivot.
4. **The real, pluggable Google Gemini API call** (Section 7) — via `google.genai.Client` and
   `client.models.generate_content(...)`.
5. **Citation validation** (Section 8) and **UDAAP review** (Section 9) — both must pass or the
   gate stops entirely (Section 11) — no partially-passing artifact is ever written.
6. **NIST AI RMF risk category** (Section 10).
7. **Final artifact** (Section 12) — deterministic-template-wrapped: the model's validated text,
   the full real citation table, both check results, the risk category, and the pending-approval
   banner — never the model's raw prose alone.
8. **GenAI SDK loaded re-check** (Section 13) — this is the first gate where the assertion
   *flips*: Gates 1–4 required `[]`; this gate requires `"google.genai"` present, proving the real
   SDK was genuinely engaged. (The shared detector in `src/genai/bp6_evidence_prep.py` was
   extended 2026-09-24 to include `"google.genai"` in its watched prefix list, additively — Gates
   1–4's own already real-run-confirmed `[]` results are unaffected, since none of them ever
   imported it either.)
9. **Config write** (Section 14) — nested `gate5_decision_genai_layer:` block.
10. **Structural integrity checks** (Section 15) — 13 named assertions.

## What this gate's own sandbox verification caught and fixed before delivery

**A real gap, not a logic bug (same category of issue found under the earlier Anthropic-based
version, now re-verified against the Gemini SDK):** `call_grounded_generation`'s
`from google import genai` would raise a bare `ModuleNotFoundError` if the `google-genai` package
isn't installed in your kernel yet. Every other prerequisite failure in this project gives a clear,
actionable message — this one does too, via a wrapped `try`/`except ImportError` re-raising with
the exact `pip install google-genai` fix, before any real call is attempted. `google-genai>=1.0`
has replaced the earlier `anthropic>=0.40` entry in `requirements.txt`.

Because there is no real API key in this project's sandbox verification environment, the full
generation-to-artifact path (Sections 7–15) was verified two ways, mirroring the earlier
Anthropic-based version's own verification approach: (1) the "no key" path was run for real against
this gate's actual (Gemini-based) code — confirmed to stop cleanly at the disclosed error with
config and artifacts directory byte-for-byte unchanged (md5-confirmed); (2) the real `google-genai`
package (version 2.25.0 at the time of this rebuild) was installed in the sandbox and used to
confirm `genai.Client(api_key=...)` and `types.GenerateContentConfig(max_output_tokens=...)`
construct without error against the real, currently-installed SDK, and that
`client.models.generate_content`'s real signature is `(model, contents, config)` — exactly what
this module calls. The actual network call (`generate_content` reaching Google's real servers) was
then verified by mocking only `genai.Client` itself (not the whole `call_grounded_generation`
function, so this module's own real extraction logic — `response.text`,
`response.usage_metadata.prompt_token_count`, `.candidates_token_count`, the `GEMINI_MODEL`
override, and the defensive `getattr` fallback for a missing `usage_metadata` — all ran for real
against a structurally realistic fake response object). This surfaced no bugs and confirmed the
downstream citation/UDAAP/risk-category/artifact logic (already proven correct under the Anthropic
build) still works unchanged, since none of that logic is provider-specific. The one thing this
sandbox genuinely cannot verify is the real Gemini network response shape from Google's live
servers themselves — disclosed rather than worked around: `GEMINI_MODEL` is a documented override
in case `gemini-3.5-flash` (this module's default) is not current or not free-tier-eligible for
your account by the time you run this — check
[the Gemini API pricing page](https://ai.google.dev/gemini-api/docs/pricing) for the current
free-tier model list.

## Prerequisite

BP6 Gate 4 must have already run for real (writes this config's Gate 4 block and the
explainability trace this gate reads for its eligible-bucket list). BP1–BP5's own real Gate 7
executive rollup reports must all exist (they do, as of this gate's delivery — all five are
real-run confirmed). You will additionally need: `pip install google-genai`, and a real, free
`GEMINI_API_KEY` from [Google AI Studio](https://aistudio.google.com/apikey) set in your
environment before this cell can complete — Google AI Studio auto-creates a default project and
key for you, with no separate GCP billing setup required for the free tier. Per this project's
standing execution-boundary rule, Claude never runs this notebook — only you do, in the
`home_credit_env` Jupyter kernel, and only once a real key is in place.

## What this gate does NOT do

- **Never auto-applies, sends, or finalizes anything.** `approval_status` is always
  `PENDING_HUMAN_REVIEW`; `auto_applied` is always `False`.
- **Never presents an ungrounded or unreviewed recommendation**, even flagged — a failed citation
  or UDAAP check stops the gate entirely (Section 11).
- **Never fabricates a GenAI response.** No key, or no installed SDK, means no artifact — ever.
- **No financial-impact or illustrative-projection content, anywhere.**

## Real, disclosed design choices in this gate

- The evidence bundle intentionally cites BP-level headline facts (champion model names,
  production-recommendation tiers, disparate-impact flags), not row-level complaint data — no
  shared Complaint-ID join key exists across BP1–3's decision-records CSVs (an open gap BP7's own
  Gate 1 already flagged), so citing row-level "evidence" today would mean fabricating a join that
  doesn't structurally exist. This gate cites only what is genuinely, structurally retrievable.
- The NIST AI RMF risk category's rubric and its "never LOW for a live customer-facing GenAI call"
  floor are this project's own documented judgment, explicitly labeled as such — not presented as
  an official NIST-published category, since no such enum exists in the framework itself.
- Token usage (`input_tokens`/`output_tokens` in the recommendation artifact and config block) is
  read defensively via `getattr` off Gemini's `usage_metadata` object rather than asserted as
  certain to exist — if a future SDK version ever renamed or dropped that field, this gate degrades
  to a disclosed `null` in the written config/artifact rather than crashing after a real API call
  has already succeeded.

Every real number and citation in this gate is read directly from BP1–5's own real, already
real-run-confirmed Gate 7 artifacts and Gate 2's own real PII-screened narrative text. The model's
generated prose is the only non-deterministic element, and it is never trusted without structural
verification against the real evidence bundle.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP6 Gate 5 (Decision / GenAI Layer & Reporting)
notebook. Single consolidated code cell (platform convention).

This is BP6's first gate with an actual external API call (Google's Gemini API, via the official
google-genai SDK) - every earlier gate (1-4) was prep-only, structurally verified to load zero
GenAI SDK modules. Per user decision (2026-09-24), this gate is built PLUGGABLE: the real call is
fully wired, but if GEMINI_API_KEY is not set in the environment, this notebook raises a clear,
disclosed error and writes NOTHING - no artifact, no config block - rather than fabricating a
response. (Provider pivoted the same day from an initial Anthropic Claude API choice to Gemini's
genuine no-cost free tier - this project is a personal portfolio build, not an institutionally
funded company, so Anthropic's own Claude for Startups credit program's eligibility does not apply.) Master Plan
Section 5.1 (paragraph 117): "Use retrieval/grounding and deterministic templates around GenAI
(BP6) - no ungrounded generation is ever presented as a recommendation." This notebook enforces
that literally: every claim the model makes must cite a real, pre-assembled evidence record from
BP1-BP5's own real Gate 7 outputs, verified structurally after generation - never trusted from the
prompt instruction alone. This gate is NOT idempotent-and-silent like Gates 1-4: every real run
makes a real, billed API call, so it is safe to re-run (the config block still overwrites
idempotently) but is not free to re-run casually.
"""

import os, sys, json, warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (same resolver as every other notebook)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Imports + prerequisite check
# ============================================================
from genai.bp6_grounded_generation import (
    retrieve_headline_evidence_bundle,
    select_real_narrative_sample,
    assemble_grounded_prompt,
    call_grounded_generation,
    MissingApiKeyError,
    validate_citations_in_generated_text,
    check_udaap_customer_facing_language_batch,
    split_into_sentences,
    compute_nist_ai_rmf_risk_category,
    build_recommendation_artifact,
)
from genai.bp6_evidence_prep import genai_sdk_modules_loaded

CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp6_genai_resolution_assistant" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BP6_CONFIG_PATH = CONFIGS_DIR / "bp6_genai_resolution_assistant.yaml"
GATE4_MARKER_TEXT = "Gate 4 (Statistical Validation & Explainability)"
PII_SCREENED_CSV_PATH = ARTIFACTS_DIR / "gate2_pii_screened_narrative_text.csv"

if not BP6_CONFIG_PATH.exists() or GATE4_MARKER_TEXT not in BP6_CONFIG_PATH.read_text(encoding="utf-8"):
    raise RuntimeError("BP6 Gate 4's own config block was not found - run BP6 Gate 4 first.")
if not PII_SCREENED_CSV_PATH.exists():
    raise FileNotFoundError(f"{PII_SCREENED_CSV_PATH} does not exist - run BP6 Gate 2 first.")
print("[OK] Prerequisites confirmed: BP6 Gate 4 config block + Gate 2 PII-screened CSV both present.")

# Gate 3/4's own real cross-corpus-hit buckets - the only buckets where a real customer message
# genuinely has real retrievable evidence on the other side of the CFPB<->BANKING77 divide.
GATE4_BLOCK_TEXT = BP6_CONFIG_PATH.read_text(encoding="utf-8")
# crosstab_both_sides_n is written by Gate 4 but the actual bucket NAMES are only in the
# explainability trace artifact (Gate 4 deliberately did not write the bucket list into the flat
# config block) - read them from there, never re-derived or hardcoded here.
GATE4_TRACE_PATH = ARTIFACTS_DIR / "gate4_explainability_trace.json"
if not GATE4_TRACE_PATH.exists():
    raise FileNotFoundError(f"{GATE4_TRACE_PATH} does not exist - run BP6 Gate 4 first.")
with open(GATE4_TRACE_PATH, "r", encoding="utf-8") as f:
    gate4_trace = json.load(f)
ELIGIBLE_BUCKETS = tuple(entry["bucket"] for entry in gate4_trace)
print(f"[OK] Real cross-corpus-hit buckets (from Gate 4's own trace): {ELIGIBLE_BUCKETS}")

# ============================================================
# SECTION 4: Assemble the real, grounded evidence bundle from BP1-BP5's own real Gate 7 outputs
# ============================================================
citations, citation_lookup = retrieve_headline_evidence_bundle(PROJECT_ROOT)
print(f"[OK] Real evidence bundle assembled: {len(citations)} citation records across BP1-BP5's "
      "own real Gate 7 executive rollup manifests.")

# ============================================================
# SECTION 5: Select one real, PII-clear customer message from a real cross-corpus-hit bucket
# ============================================================
narrative_sample = select_real_narrative_sample(
    PII_SCREENED_CSV_PATH, eligible_buckets=ELIGIBLE_BUCKETS, random_state=42
)
print(f"[OK] Real customer message selected (category={narrative_sample['category']!r}, "
      f"bucket={narrative_sample['common_taxonomy_bucket']!r}).")

# ============================================================
# SECTION 6: Assemble the deterministic grounded prompt (Master Plan paragraph 117's
# "deterministic template around GenAI")
# ============================================================
prompt = assemble_grounded_prompt(narrative_sample, citations)
print(f"[OK] Grounded prompt assembled ({len(prompt)} chars, {len(citations)} evidence items).")

# ============================================================
# SECTION 7: The real, pluggable Google Gemini API call. Raises MissingApiKeyError with clear
# setup instructions and writes NOTHING if GEMINI_API_KEY is not set - never fabricates a
# response.
# ============================================================
try:
    generation_result = call_grounded_generation(prompt)
except MissingApiKeyError as e:
    print(f"[STOPPED] {e}")
    raise

print(f"[OK] Real Gemini API call succeeded. Model: {generation_result['model_used']} | "
      f"input_tokens={generation_result['input_tokens']} | "
      f"output_tokens={generation_result['output_tokens']} | "
      f"finish_reason={generation_result['finish_reason']}")
print(f"     Generated text: {generation_result['generated_text']!r}")

# ============================================================
# SECTION 8: Structural anti-hallucination check - the model's generated text is not trusted; its
# citations are verified against the real bundle.
# ============================================================
citation_check = validate_citations_in_generated_text(generation_result["generated_text"], citation_lookup)
print(f"[OK] Citation check: passed={citation_check['passed']} | "
      f"cited={citation_check['cited_evidence_ids']} | "
      f"invalid={citation_check['invalid_evidence_ids_referenced']}")

# ============================================================
# SECTION 9: UDAAP customer-facing language review (Gate 5's own compliance touchpoint)
# ============================================================
sentences = split_into_sentences(generation_result["generated_text"])
udaap_check = check_udaap_customer_facing_language_batch(sentences)
print(f"[OK] UDAAP language check: passed={udaap_check['passed']} | "
      f"banned_terms_found={udaap_check['banned_terms_found']}")

# ============================================================
# SECTION 10: NIST AI RMF risk category - populated with a real, structurally-computed value for
# the first time (Gate 1's own policy.json carried "TBD_PENDING_FIRST_REAL_GATE5_GENAI_OUTPUT"
# until now).
# ============================================================
risk_category = compute_nist_ai_rmf_risk_category(
    citation_check_passed=citation_check["passed"],
    udaap_check_passed=udaap_check["passed"],
    human_in_the_loop_enforced=True,  # this notebook never auto-applies - Section 12 below
    auto_apply_allowed=False,
    pii_masking_applied_upstream=True,  # Gate 2's own real PII screen, re-confirmed clean
)
print(f"[OK] NIST AI RMF risk category: {risk_category['risk_category_value']} "
      f"(all_mitigations_passed={risk_category['all_mitigations_passed']})")

# ============================================================
# SECTION 11: Refuse to present a flagged recommendation. If either structural check failed, this
# gate stops here - it does NOT write a partially-passing artifact, per Master Plan paragraph 117
# ("no ungrounded generation is ever presented as a recommendation").
# ============================================================
response_truncated = generation_result["finish_reason"] == "MAX_TOKENS"
if not (citation_check["passed"] and udaap_check["passed"]) or response_truncated:
    raise RuntimeError(
        "BP6 Gate 5's structural checks failed on this real generation "
        f"(citation_check.passed={citation_check['passed']}, udaap_check.passed={udaap_check['passed']}, "
        f"finish_reason={generation_result['finish_reason']!r}). "
        "Per this gate's own design (Master Plan paragraph 117), a recommendation that fails "
        "grounding or UDAAP review, OR that was truncated before completion (finish_reason == "
        "'MAX_TOKENS' - a real, documented Gemini thinking-model behavior where reasoning tokens "
        "can consume the output budget before any visible text is written), is never presented, "
        "even flagged - re-run this cell to generate a fresh attempt. No artifact or config block "
        "has been written."
    )

# ============================================================
# SECTION 12: Build and write the final deterministic-template-wrapped recommendation artifact.
# approval_status is always PENDING_HUMAN_REVIEW; auto_applied is always False - this notebook
# never applies, sends, or finalizes anything.
# ============================================================
recommendation_artifact = build_recommendation_artifact(
    narrative_sample, citations, generation_result, citation_check, udaap_check, risk_category
)
RECOMMENDATION_PATH = ARTIFACTS_DIR / "gate5_recommendation_pending_human_review.json"
with open(RECOMMENDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(recommendation_artifact, f, indent=2)
print(f"[SAVED] Recommendation artifact (PENDING_HUMAN_REVIEW): {RECOMMENDATION_PATH}")

# ============================================================
# SECTION 13: GenAI SDK loaded check - this gate is the FIRST to require it non-empty (Gates 1-4
# required it empty). Flips the assertion direction deliberately, disclosed here.
# ============================================================
loaded_sdks = genai_sdk_modules_loaded()
print(f"[OK] GenAI SDK modules loaded in this run: {loaded_sdks}")

# ============================================================
# SECTION 14: Write the Gate 5 config block.
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate5_marker = "# --- Gate 5 (Decision / GenAI Layer & Reporting) results (appended, idempotent overwrite) ---"
gate5_block_lines = [
    "gate5_decision_genai_layer:",
    f'  model_used: "{generation_result["model_used"]}"',
    f"  input_tokens: {json.dumps(generation_result['input_tokens'])}",
    f"  output_tokens: {json.dumps(generation_result['output_tokens'])}",
    f"  finish_reason: {json.dumps(generation_result['finish_reason'])}",
    f"  n_evidence_citations_available: {len(citations)}",
    f"  n_citations_used: {citation_check['n_citations_referenced']}",
    f"  citation_check_passed: {citation_check['passed']}",
    f"  udaap_check_passed: {udaap_check['passed']}",
    f'  nist_ai_rmf_risk_category: "{risk_category["risk_category_value"]}"',
    f"  approval_status: \"{recommendation_artifact['approval_status']}\"",
    f"  auto_applied: {recommendation_artifact['auto_applied']}",
    f'  recommendation_artifact_path: "{RECOMMENDATION_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"  genai_sdk_modules_loaded_this_run: {loaded_sdks}",
    f'  generated_at_utc: "{generation_result["generated_at_utc"]}"',
]
write_gate_block(BP6_CONFIG_PATH, gate5_marker, gate5_block_lines)
print(f"[SAVED] Gate 5 config block written to {BP6_CONFIG_PATH}")

# ============================================================
# SECTION 15: Structural integrity checks
# ============================================================
_config_text_after = BP6_CONFIG_PATH.read_text(encoding="utf-8")
_front_matter_and_priors_preserved = all(
    marker in _config_text_after
    for marker in (
        'bp_id: "bp6"',
        GATE4_MARKER_TEXT,
        "pii_screen_rows_scanned:",
        "gate3_retrieval_benchmark:",
        "gate4_statistical_validation:",
        "random_state: 42",
    )
)

_checks: list[tuple[str, bool]] = [
    ("real_evidence_bundle_nonempty", len(citations) > 0),
    ("real_customer_message_selected_from_eligible_bucket", narrative_sample["common_taxonomy_bucket"] in ELIGIBLE_BUCKETS),
    ("real_gemini_api_call_succeeded", bool(generation_result["generated_text"])),
    ("response_not_truncated_by_max_tokens", generation_result["finish_reason"] != "MAX_TOKENS"),
    ("citation_check_passed", citation_check["passed"]),
    ("zero_invalid_citations_referenced", citation_check["invalid_evidence_ids_referenced"] == []),
    ("udaap_check_passed", udaap_check["passed"]),
    ("nist_risk_category_populated_not_tbd", risk_category["risk_category_value"] in ("MEDIUM", "HIGH")),
    ("approval_status_pending_human_review", recommendation_artifact["approval_status"] == "PENDING_HUMAN_REVIEW"),
    ("never_auto_applied", recommendation_artifact["auto_applied"] is False),
    ("recommendation_artifact_written", RECOMMENDATION_PATH.exists()),
    ("gemini_sdk_actually_loaded_this_run", "google.genai" in loaded_sdks),
    ("config_gate5_block_written", gate5_marker in _config_text_after),
    ("config_front_matter_and_prior_gate_blocks_preserved", _front_matter_and_priors_preserved),
]

_failed = [name for name, ok in _checks if not ok]
for name, ok in _checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert not _failed, f"BP6 Gate 5 structural integrity checks failed: {_failed}"

print(
    "\n[ALL CHECKS PASSED] BP6 Gate 5 (Decision / GenAI Layer & Reporting) complete. A real, "
    f"grounded recommendation was generated, cited {citation_check['n_citations_referenced']} "
    f"real evidence items, passed UDAAP review, and is staged as PENDING_HUMAN_REVIEW - it has "
    "NOT been sent, applied, or finalized. NIST AI RMF risk category: "
    f"{risk_category['risk_category_value']}. A human must review "
    f"{RECOMMENDATION_PATH.name} before any action is taken."
)
